In [1]:
#feature eng
import pandas as pd
import os

DATA_DIR = r'C:\Projects\ecommerce-bi-dashboard\data\raw'
PROCESSED_DIR = r'C:\Projects\ecommerce-bi-dashboard\data\processed'

orders = pd.read_csv(f'{PROCESSED_DIR}\\orders_cleaned.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
products = pd.read_csv(f'{PROCESSED_DIR}\\products_cleaned.csv')
items = pd.read_csv(f'{DATA_DIR}\\olist_order_items_dataset.csv')
customers = pd.read_csv(f'{DATA_DIR}\\olist_customers_dataset.csv')
sellers = pd.read_csv(f'{DATA_DIR}\\olist_sellers_dataset.csv')
payments = pd.read_csv(f'{DATA_DIR}\\olist_order_payments_dataset.csv')
reviews = pd.read_csv(f'{DATA_DIR}\\olist_order_reviews_dataset.csv')

print(f"orders: {orders.shape}, products: {products.shape}")

orders: (96478, 10), products: (32951, 10)


In [2]:
def delay_bucket(days):
    if days <= 0:
        return 'On time'
    elif days <= 3:
        return '1-3 days late'
    elif days <= 7:
        return '4-7 days late'
    else:
        return '8+ days late'

orders['delay_bucket'] = orders['delivery_delay_days'].apply(delay_bucket)

orders = orders.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

order_value = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum')
).reset_index()
orders = orders.merge(order_value, on='order_id', how='left')
orders['order_total'] = orders['total_price'] + orders['total_freight']

orders = orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
    on='customer_id', how='left'
)
orders['order_rank'] = orders.groupby('customer_unique_id')['order_purchase_timestamp'].rank(method='first')
orders['is_repeat_customer'] = (orders.groupby('customer_unique_id')['order_id'].transform('count') > 1).astype(int)

print(orders[['order_id', 'delay_bucket', 'review_score', 'order_total', 'order_rank']].head())

                           order_id delay_bucket  review_score  order_total  \
0  e481f51cbdc54678b7cc49136f2d6af7      On time           4.0        38.71   
1  53cdb2fc8bc7dce0b6741e2150273451      On time           4.0       141.46   
2  47770eb9100c2d0c44946d9cf07ec65d      On time           5.0       179.12   
3  949d5b44dbf5de918fe9c16f97b45f8a      On time           5.0        72.20   
4  ad21c59c0840e6cb83a9ceb5573f8159      On time           5.0        28.62   

   order_rank  
0         2.0  
1         1.0  
2         1.0  
3         1.0  
4         1.0  


In [3]:
payment_summary = payments.groupby('order_id').agg(
    max_installments=('payment_installments', 'max'),
    payment_type=('payment_type', 'first'),
    total_payment=('payment_value', 'sum')
).reset_index()

def installment_bucket(n):
    if n <= 1:
        return '1x'
    elif n <= 4:
        return '2-4x'
    elif n <= 8:
        return '5-8x'
    else:
        return '9x+'

payment_summary['installment_bucket'] = payment_summary['max_installments'].apply(installment_bucket)

orders = orders.merge(payment_summary, on='order_id', how='left')

In [4]:
order_seller = items[['order_id', 'seller_id']].drop_duplicates()
seller_orders = order_seller.merge(orders[['order_id', 'is_late', 'review_score', 'order_total']], on='order_id', how='left')

seller_summary = seller_orders.groupby('seller_id').agg(
    total_orders=('order_id', 'nunique'),
    late_orders=('is_late', 'sum'),
    avg_review=('review_score', 'mean'),
    total_revenue=('order_total', 'sum')
).reset_index()

seller_summary['late_rate'] = seller_summary['late_orders'] / seller_summary['total_orders']
seller_summary = seller_summary.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

seller_summary = seller_summary.sort_values('late_rate', ascending=False)
print(seller_summary.head(10))

                             seller_id  total_orders  late_orders  avg_review  \
394   20f0aeea30bc3b8c4420be8ced4226c0             1          1.0         NaN   
733   3da38366e7bd9baf6369071f782ecdf0             1          1.0         1.0   
2292  be1e9e378700cecaa4ebf71433d7915c             2          2.0         3.0   
2699  df683dfda87bf71ac3fc63063fba369d             1          1.0         1.0   
2274  bcd2d7510d58e293f20fad6438c1b314             1          1.0         NaN   
2330  c13ef0cfbe42f190780f621ce81f2234             1          1.0         3.0   
2393  c611f4ce9ce875bcc063fa97fd4d7d12             1          1.0         4.0   
2416  c7b7db6c8f3c64a7cc1afa634db21d50             1          1.0         1.0   
2453  ca5832c6960267b71041f74bb39e8b12             1          1.0         1.0   
295   19484c79cef6c062cb177aa4ef2fcc3c             1          1.0         1.0   

      total_revenue  late_rate seller_state  
394           29.42        1.0           SP  
733           86

In [5]:
order_product = items[['order_id', 'product_id']].drop_duplicates()
order_product = order_product.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')
order_product['product_category_name_english'] = order_product['product_category_name_english'].fillna('unknown')

orders_with_category = orders.merge(order_product, on='order_id', how='left')

category_summary = orders_with_category.groupby('product_category_name_english').agg(
    total_orders=('order_id', 'nunique'),
    avg_review=('review_score', 'mean'),
    late_rate=('is_late', 'mean'),
    total_revenue=('order_total', 'mean')
).reset_index().sort_values('total_orders', ascending=False)

print(category_summary.head(10))

   product_category_name_english  total_orders  avg_review  late_rate  \
7                 bed_bath_table          9272    3.941001   0.072760   
43                 health_beauty          8647    4.223995   0.074412   
65                sports_leisure          7530    4.214304   0.064370   
15         computers_accessories          6530    4.053084   0.062998   
39               furniture_decor          6307    4.025741   0.068837   
49                    housewares          5743    4.176471   0.052712   
71                 watches_gifts          5495    4.100248   0.072711   
68                     telephony          4093    4.032483   0.070780   
5                           auto          3810    4.129967   0.071684   
69                          toys          3804    4.228188   0.063525   

    total_revenue  
7      144.081281  
43     165.456291  
65     151.745896  
15     166.666540  
39     150.599657  
49     136.705983  
71     230.847224  
68      93.543085  
5      179.28604

In [6]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

orders_with_category.to_csv(f'{PROCESSED_DIR}\\fact_orders.csv', index=False)
seller_summary.to_csv(f'{PROCESSED_DIR}\\dim_seller_summary.csv', index=False)
category_summary.to_csv(f'{PROCESSED_DIR}\\dim_category_summary.csv', index=False)
customers.to_csv(f'{PROCESSED_DIR}\\dim_customers.csv', index=False)

print("Day 2 complete — all processed files saved")
print(f"fact_orders: {orders_with_category.shape}")

Day 2 complete — all processed files saved
fact_orders: (100777, 26)


In [7]:
print(orders.columns.tolist())
print('order_id' in orders.columns)

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'is_late', 'delay_bucket', 'review_score', 'total_price', 'total_freight', 'order_total', 'customer_unique_id', 'customer_state', 'customer_city', 'order_rank', 'is_repeat_customer', 'max_installments', 'payment_type', 'total_payment', 'installment_bucket']
True


In [8]:
print('order_id' in orders_with_category.columns)
print('installment_bucket' in orders_with_category.columns)
print([c for c in orders_with_category.columns if c.endswith('_x') or c.endswith('_y')])

True
True
[]


In [10]:
print("Null timestamps in fact_orders:")
print(orders_with_category['order_purchase_timestamp'].isnull().sum())
print("\nDate range:")
print(f"Min: {orders_with_category['order_purchase_timestamp'].min()}")
print(f"Max: {orders_with_category['order_purchase_timestamp'].max()}")
print("\nUnique months:")
print(orders_with_category['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index())

Null timestamps in fact_orders:
0

Date range:
Min: 2016-09-15 12:16:38
Max: 2018-08-29 15:00:37

Unique months:
order_purchase_timestamp
2016-09       1
2016-10     281
2016-12       1
2017-01     796
2017-02    1721
2017-03    2650
2017-04    2382
2017-05    3726
2017-06    3280
2017-07    4079
2017-08    4411
2017-09    4327
2017-10    4713
2017-11    7673
2017-12    5738
2018-01    7406
2018-02    6823
2018-03    7272
2018-04    7103
2018-05    7019
2018-06    6348
2018-07    6398
2018-08    6629
Freq: M, Name: count, dtype: int64
